In [ ]:
import argparse
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

def preprocessing(test_fname):
    train_data = pd.read_csv('train.csv')
    test_data = pd.read_csv(test_fname)

    target_mapping = {'low': 0, 'medium': 1, 'high': 2}
    train_data['Target'] = train_data['Target'].map(target_mapping)
    y = train_data['Target']
    train_data = train_data.drop(columns=['Target'])

    uid_train = train_data['UID']
    uid_test = test_data['UID']
    train_data = train_data.drop(columns=['UID'])
    test_data = test_data.drop(columns=['UID'])

    columns_to_remove = [
        'RawLocationId', 'TownId', 'DistrictId', 'FarmingCommunityId', 
        'AgriculturalPostalZone', 'WaterAccessPointsCalc', 
        'PrimaryCropAreaSqft2', 'CultivatedAreaSqft1'
    ]
    train_data = train_data.drop(columns=columns_to_remove)
    test_data = test_data.drop(columns=columns_to_remove)

    missing_percent = train_data.isnull().mean() * 100
    columns_to_drop = missing_percent[missing_percent > 80].index
    
    train_data = train_data.drop(columns=columns_to_drop)
    test_data = test_data.drop(columns=columns_to_drop, errors='ignore')

    for col in train_data.columns:
        if train_data[col].nunique() > 200:
            mean_value = train_data[col].median()
            train_data[col].fillna(mean_value, inplace=True)
            test_data[col].fillna(mean_value, inplace=True)
        else:
            mode_value = train_data[col].mode()[0]
            train_data[col].fillna(mode_value, inplace=True)
            test_data[col].fillna(mode_value, inplace=True)

    return train_data, y, test_data, uid_test

def train_model(train_data, test_data, y):
    xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42)
    X = train_data

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

    threshold_low_range = np.arange(0.22, 0.31, 0.01)
    threshold_high_diff = np.arange(0.04, 0.09, 0.01)
    
    xgb_model.fit(X_train, y_train)
    
    y_prob = xgb_model.predict_proba(X_test)
    y_pred_base = y_prob.argmax(axis=1)
    
    best_f1_macro = 0
    best_threshold_low = 0
    best_threshold_high = 0
    
    for threshold_low in threshold_low_range:
        for diff in threshold_high_diff:
            threshold_high = min(threshold_low + diff, 0.30)
            
            y_pred_adjusted = y_pred_base.copy()
            
            for i in range(len(y_pred_adjusted)):
                if y_pred_adjusted[i] == 1:
                    if y_prob[i, 0] > threshold_high:
                        y_pred_adjusted[i] = 0 
                    elif y_prob[i, 2] > threshold_low:
                        y_pred_adjusted[i] = 2
            
            f1_macro = f1_score(y_test, y_pred_adjusted, average='macro')
            
            if f1_macro > best_f1_macro:
                best_f1_macro = f1_macro
                best_threshold_low = threshold_low
                best_threshold_high = threshold_high

    y_prob = xgb_model.predict_proba(test_data)
    y_pred = y_prob.argmax(axis=1)
    for i in range(len(y_pred)):
        if y_pred[i] == 1:
            if y_prob[i, 0] > best_threshold_low:
                y_pred[i] = 0
            elif y_prob[i, 2] > best_threshold_high:
                y_pred[i] = 2
    
    return y_pred

def make_predictions(test_fname, predictions_fname):
    train_data, y, test_data, uid_test = preprocessing(test_fname)
    test_predictions = train_model(train_data, test_data, y)
    
    reverse_target_mapping = {0: 'low', 1: 'medium', 2: 'high'}
    test_predictions_labels = [reverse_target_mapping[pred] for pred in test_predictions]
    
    submission = pd.DataFrame({
        'UID': uid_test,
        'Target': test_predictions_labels
    })
    
    submission.to_csv(predictions_fname, index=False)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-file", type=str, help="file path of train.csv")
    parser.add_argument("--test-file", type=str, help="file path of test.csv")
    parser.add_argument("--predictions-file", type=str, help="save path of predictions")
    args = parser.parse_args()
    make_predictions(args.test_file, args.predictions_file)
